# Titans-style Memory Adapter trên Qwen2.5-0.5B (tiếng Việt)

**Mục tiêu:** minh họa *học tại thời điểm test* (test-time memorization) của bài báo *Titans: Learning to Memorize at Test Time*.

**Lưu ý trung thực:** đây **không phải** Titans end-to-end của paper. Backbone Qwen đã pretrain được **đóng băng**; ta gắn thêm một module `NeuralMemory` (thư viện `titans-pytorch`) vào hidden state ở giữa mạng và chỉ train phần mới. Hãy gọi nó là *"memory adapter kiểu Titans trên backbone đóng băng"*.

| Pha | Việc gì xảy ra | Tham số thay đổi |
|---|---|---|
| 1. Pretrain memory (outer loop) | Train trên Wikipedia tiếng Việt, tài liệu dài chia thành segment liên tiếp; memory phải mang thông tin sang segment sau | `mem` (K/V/Q, cổng θ/η/α, khởi tạo), `out`, `gate` |
| 2. Test time (inner loop) | Đóng băng **mọi** tham số; dữ liệu mới đi qua từng segment, chỉ **trọng số fast-weight của memory** cập nhật | Chỉ trạng thái memory |

**Cách chạy trên Kaggle:** Accelerator = GPU (T4/P100), Internet = On, Run All. Checkpoint lưu ở `/kaggle/working/ckpt/` (`last.pt` để resume, `best.pt`, `titans_vi_memory.pt` để dùng cho webapp). Nếu hết phiên: *Save Version* → thêm output làm Input ở phiên sau → đặt `INIT_CKPT` trỏ tới `last.pt`.

**Chế độ thử nhanh:** đặt `DEBUG = True` để chạy cả notebook bằng model ngẫu nhiên rất nhỏ (không cần GPU/Internet) nhằm kiểm tra code.

In [ ]:
!pip install -q titans-pytorch transformers datasets accelerate matplotlib

In [ ]:
import os, time, math, json, re, random, html as _html
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

DEBUG = False   # True = chạy thử nhanh bằng model ngẫu nhiên nhỏ, không cần GPU/Internet

# ---- Backbone & dữ liệu ----
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
SEG_LEN    = 256      # độ dài 1 segment = cửa sổ mà backbone nhìn thấy
N_SEGS     = 8        # số segment liên tiếp của 1 tài liệu khi train
TBPTT      = 2        # backward sau mỗi TBPTT segment (cắt gradient qua state memory)
N_TRAIN_DOCS = 6000
N_EVAL_DOCS  = 24
# Loại các bài trùng chủ đề vật lý khỏi dữ liệu train (tránh rò rỉ khi test bằng sách vật lý)
EXCLUDE_TITLE_REGEX = r"vật lý|cơ học|nhiệt động|điện từ|lượng tử|tương đối|newton|động lực học|năng lượng"

# ---- Memory ----
MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD, MEM_DEPTH = 16, 4, 64, 2
USE_MOMENTUM = True     # đặt False để chạy ablation (train lại) không có momentum
HOOK_LAYER   = None     # None = giữa mạng

# ---- Train ----
BATCH_DOCS, LR, WARMUP, MAX_STEPS = 2, 1e-3, 100, 3000
EVAL_EVERY, SAVE_EVERY, MAX_MINUTES = 200, 100, 170
CKPT_DIR  = os.environ.get("CKPT_DIR", "/kaggle/working/ckpt")
DATA_PATH = os.environ.get("DATA_PATH", "/kaggle/working/data_vi.pt")
INIT_CKPT = None        # đường dẫn last.pt từ phiên trước (nếu muốn tiếp tục)
SEED = 0

if DEBUG:
    SEG_LEN, N_SEGS, TBPTT = 32, 4, 2
    MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD = 4, 2, 16
    N_TRAIN_DOCS, N_EVAL_DOCS, MAX_STEPS = 16, 4, 6
    WARMUP, EVAL_EVERY, SAVE_EVERY = 2, 3, 3

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| DEBUG:", DEBUG)
try:
    from IPython.display import HTML, display
except Exception:
    HTML = lambda s: s
    display = print

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Qwen2Config, Qwen2ForCausalLM

if DEBUG:
    _cfg = Qwen2Config(vocab_size=500, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                       num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=4096)
    lm = Qwen2ForCausalLM(_cfg)
    def encode(t): return [b % 500 for b in t.encode("utf-8")]
    def decode(ids): return "".join(chr(32 + (int(i) % 90)) for i in ids)
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    _dt = torch.float16 if device == "cuda" else torch.float32
    try:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=_dt)
    except TypeError:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=_dt)
    def encode(t): return tok(t, add_special_tokens=False)["input_ids"]
    def decode(ids): return tok.decode(ids)

lm = lm.to(device).eval()
for p in lm.parameters():
    p.requires_grad_(False)
print("tham số backbone (đóng băng), triệu:", round(sum(p.numel() for p in lm.parameters()) / 1e6, 1))

## 1. Dữ liệu pretrain cho memory: Wikipedia tiếng Việt

Tài liệu dài, cắt đúng `SEG_LEN * N_SEGS` token. Các bài có tiêu đề trùng chủ đề vật lý bị loại (xem `EXCLUDE_TITLE_REGEX`). Mười mấy bài cuối làm tập đánh giá, **không train**.

In [ ]:
def build_dataset():
    from datasets import load_dataset
    need = SEG_LEN * N_SEGS
    ds = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=10000)
    rx = re.compile(EXCLUDE_TITLE_REGEX, re.I)
    out, seen = [], 0
    for ex in ds:
        seen += 1
        if rx.search(ex["title"]) or len(ex["text"]) < need * 2:
            continue
        ids = encode(ex["text"])
        if len(ids) < need:
            continue
        out.append(torch.tensor(ids[:need]))
        if len(out) % 500 == 0:
            print(len(out), "tài liệu (đã duyệt", seen, "bài)")
        if len(out) >= N_TRAIN_DOCS + N_EVAL_DOCS:
            break
    return torch.stack(out)

if DEBUG:
    data = torch.randint(0, lm.config.vocab_size, (N_TRAIN_DOCS + N_EVAL_DOCS, SEG_LEN * N_SEGS))
elif os.path.exists(DATA_PATH):
    data = torch.load(DATA_PATH)
else:
    data = build_dataset()
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    torch.save(data, DATA_PATH)

train_docs, eval_docs = data[:-N_EVAL_DOCS], data[-N_EVAL_DOCS:]
print("train:", tuple(train_docs.shape), "| eval:", tuple(eval_docs.shape),
      "| tổng token train (triệu):", round(train_docs.numel() / 1e6, 1))

## 2. Mô hình: backbone đóng băng + Neural Memory kiểu Titans

Memory được gắn bằng *forward hook* vào đầu ra của một layer giữa mạng. Ở mỗi segment: hidden state của segment được **ghi** vào memory (cập nhật fast-weight bằng gradient của `‖M(k)−v‖²` có momentum và weight decay/forgetting, đúng Eq. 12–14), rồi memory **đọc** lại và cộng vào hidden state qua `out` và `gate`. Việc đọc dùng trọng số memory của các chunk *trước* nên đúng tính nhân quả. Backbone mỗi lần chỉ thấy `SEG_LEN` token; mọi thông tin xa hơn phải đi qua memory.

In [ ]:
from titans_pytorch import NeuralMemory, mem_state_detach

class MemLM(nn.Module):
    def __init__(self, lm, layer=None):
        super().__init__()
        self.lm = lm
        d = lm.config.hidden_size
        # batch_size=MEM_CHUNK: sau mỗi chunk memory mới được cập nhật và chunk sau tính gradient theo trọng số MỚI
        # (mini-batch gradient descent của paper). Nếu để batch_size=None, thư viện luôn tính surprise theo trọng số
        # khởi tạo nên memory không bao giờ ảnh hưởng tới surprise.
        self.mem = NeuralMemory(dim=d, chunk_size=MEM_CHUNK, batch_size=MEM_CHUNK, heads=MEM_HEADS, dim_head=MEM_DIM_HEAD,
                                momentum=USE_MOMENTUM,
                                default_model_kwargs=dict(depth=MEM_DEPTH, expansion_factor=4.0))
        # titans-pytorch tạo tham số khởi tạo memory theo head bằng `repeat` (view stride 0) -> Adam không ghi được.
        # Thay bằng bản sao liên tục, giá trị giữ nguyên.
        for i, p in enumerate(self.mem.memory_model_parameters):
            self.mem.memory_model_parameters[i] = nn.Parameter(p.detach().clone().contiguous())
        self.out = nn.Linear(d, d, bias=False)
        nn.init.normal_(self.out.weight, std=0.02)
        self.gate = nn.Parameter(torch.tensor(0.5))
        self.layer = layer if layer is not None else lm.config.num_hidden_layers // 2
        self.state, self.enabled, self.trace = None, True, False
        self.calls, self.w_ref = [], None
        lm.model.layers[self.layer].register_forward_hook(self._hook)

    def reset(self):
        self.state = None

    def detach_state(self):
        if self.state is not None:
            self.state = mem_state_detach(self.state)

    def trainable_parameters(self):
        return list(self.mem.parameters()) + list(self.out.parameters()) + [self.gate]

    def _hook(self, module, args, output):
        if not self.enabled:
            return output
        h = output[0] if isinstance(output, tuple) else output
        r, st, sur = self.mem(h.float(), state=self.state, return_surprises=True)
        self.state = st
        if self.trace:
            with torch.no_grad():
                loss_t, lr_t = sur
                flat = torch.cat([v.flatten(2) for k, v in st.updates.items()
                                  if k.startswith("model.weights")], dim=-1).float()
                if self.w_ref is None:
                    self.w_ref = flat[:, 0].clone()
                dist = ((flat - self.w_ref[:, None]).norm(dim=-1) / self.w_ref.norm(dim=-1, keepdim=True)).mean(0)
                step = (flat[:, 1:] - flat[:, :-1]).norm(dim=-1).mean(0)
                self.calls.append(dict(
                    surprise=loss_t.float().mean((0, 1)).cpu().numpy(),
                    theta=lr_t.float().mean((0, 1)).cpu().numpy(),
                    dist=dist.cpu().numpy(), step=step.cpu().numpy()))
        h2 = h + (self.gate * self.out(r)).to(h.dtype)
        return (h2,) + tuple(output[1:]) if isinstance(output, tuple) else h2

model = MemLM(lm, HOOK_LAYER)
model.mem.to(device); model.out.to(device); model.gate.data = model.gate.data.to(device)
n_train = sum(p.numel() for p in model.trainable_parameters())
print(f"hook ở layer {model.layer}/{lm.config.num_hidden_layers} | tham số được train (triệu): {n_train/1e6:.2f}")

## 3. Hàm loss, đánh giá và checkpoint

In [ ]:
def doc_losses(model, ids, train=False, scaler=None, reset_each=False):
    # ids: (B, N_SEGS*SEG_LEN). Trả về list loss trung bình từng segment.
    per_seg = []
    model.reset()
    for g in range(0, N_SEGS, TBPTT):
        group_loss = 0.0
        for s in range(g, min(g + TBPTT, N_SEGS)):
            if reset_each:
                model.reset()
            seg = ids[:, s * SEG_LEN:(s + 1) * SEG_LEN]
            logits = model.lm(input_ids=seg, use_cache=False).logits
            l = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                                seg[:, 1:].reshape(-1))
            per_seg.append(l.item())
            group_loss = group_loss + l / N_SEGS
        if train:
            scaler.scale(group_loss).backward()
        model.detach_state()
    return per_seg


@torch.no_grad()
def evaluate(model, docs):
    acc = {"mem": [], "reset": [], "base": []}
    for i in range(0, len(docs), BATCH_DOCS):
        ids = docs[i:i + BATCH_DOCS].to(device)
        model.enabled = True
        acc["mem"].append(doc_losses(model, ids))
        acc["reset"].append(doc_losses(model, ids, reset_each=True))
        model.enabled = False
        acc["base"].append(doc_losses(model, ids))
        model.enabled = True
    res = {k: np.mean(v, axis=0) for k, v in acc.items()}      # theo segment
    summ = {k: float(v[1:].mean()) for k, v in res.items()}     # bỏ segment đầu (chưa có quá khứ)
    return summ, res


CFG = dict(model_name=MODEL_NAME, seg_len=SEG_LEN, n_segs=N_SEGS, mem_chunk=MEM_CHUNK, mem_heads=MEM_HEADS,
           mem_dim_head=MEM_DIM_HEAD, mem_depth=MEM_DEPTH, use_momentum=USE_MOMENTUM, layer=model.layer)

params = model.trainable_parameters()
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

def save_ckpt(name, step, best, with_opt=True):
    os.makedirs(CKPT_DIR, exist_ok=True)
    blob = {"mem": model.mem.state_dict(), "out": model.out.state_dict(), "gate": model.gate.data.cpu(),
            "step": step, "best": best, "cfg": CFG}
    if with_opt:
        blob["opt"] = opt.state_dict(); blob["scaler"] = scaler.state_dict()
    path = os.path.join(CKPT_DIR, name)
    torch.save(blob, path + ".tmp")
    os.replace(path + ".tmp", path)

def load_ckpt(path, with_opt=True):
    blob = torch.load(path, map_location=device)
    model.mem.load_state_dict(blob["mem"]); model.out.load_state_dict(blob["out"])
    model.gate.data = blob["gate"].to(device)
    if with_opt and "opt" in blob:
        opt.load_state_dict(blob["opt"]); scaler.load_state_dict(blob["scaler"])
    return blob["step"], blob["best"]

start_step, best_val = 0, float("inf")
_resume = INIT_CKPT or os.path.join(CKPT_DIR, "last.pt")
if os.path.exists(_resume):
    start_step, best_val = load_ckpt(_resume)
    print("Đã resume từ", _resume, "| bước", start_step, "| best", best_val)
else:
    print("Bắt đầu từ đầu.")

## 4. Pretrain phần memory (outer loop)

Backbone không đổi. Loss là cross-entropy dự đoán token tiếp theo trên từng segment; để giảm loss ở segment sau, memory buộc phải ghi và truy xuất thông tin từ các segment trước. Checkpoint `last.pt` lưu định kỳ (có optimizer để resume), `best.pt` lưu theo loss đánh giá thấp nhất.

In [ ]:
def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    p = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, p))))

t0 = time.time()
model.mem.train(); model.out.train()
log_loss, step = [], start_step
order = torch.randperm(len(train_docs))
while step < MAX_STEPS:
    if (time.time() - t0) / 60 > MAX_MINUTES:
        print("Hết thời gian cho phép, dừng và lưu checkpoint.")
        break
    idx = order[(step * BATCH_DOCS) % len(train_docs):][:BATCH_DOCS]
    if len(idx) < BATCH_DOCS:
        idx = torch.randint(0, len(train_docs), (BATCH_DOCS,))
    ids = train_docs[idx].to(device)

    for g in opt.param_groups:
        g["lr"] = lr_at(step)
    opt.zero_grad(set_to_none=True)
    per_seg = doc_losses(model, ids, train=True, scaler=scaler)
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(opt); scaler.update()
    step += 1
    log_loss.append(float(np.mean(per_seg[1:])))

    if step % 10 == 0 or step == 1:
        print(f"bước {step}/{MAX_STEPS} | loss (seg>=1) {np.mean(log_loss[-10:]):.4f} | {(time.time()-t0)/60:.1f} phút")
    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        model.mem.eval(); model.out.eval()
        summ, _ = evaluate(model, eval_docs)
        print(f"  [eval] mem {summ['mem']:.4f} | reset {summ['reset']:.4f} | backbone thuần {summ['base']:.4f}"
              f" | lợi ích của quá khứ qua memory (reset-mem): {summ['reset']-summ['mem']:+.4f}")
        if summ["mem"] < best_val:
            best_val = summ["mem"]; save_ckpt("best.pt", step, best_val, with_opt=False)
        model.mem.train(); model.out.train()
    if step % SAVE_EVERY == 0:
        save_ckpt("last.pt", step, best_val)

save_ckpt("last.pt", step, best_val)
save_ckpt("titans_vi_memory.pt", step, best_val, with_opt=False)
model.mem.eval(); model.out.eval()
print("Đã lưu checkpoint tại", CKPT_DIR, "| bước", step)

## 5. Đánh giá: memory có giúp không?

Ba cấu hình trên cùng tập đánh giá chưa từng train:
- **mem**: memory mang trạng thái qua các segment (Titans-style).
- **reset**: cùng model nhưng xóa memory ở mỗi segment, tức không có quá khứ. Hiệu `reset − mem` là phần lợi ích thực sự do thông tin quá khứ.
- **base**: Qwen thuần, chỉ nhìn một segment (Transformer cùng cửa sổ).

In [ ]:
import matplotlib
if DEBUG: matplotlib.use("Agg")
import matplotlib.pyplot as plt

summ, per_seg = evaluate(model, eval_docs)
print({k: round(v, 4) for k, v in summ.items()}, "(trung bình trên các segment từ thứ 2)")
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, lab in [("base", "Qwen thuần (cửa sổ 1 segment)"), ("reset", "Có adapter, xóa memory mỗi segment"), ("mem", "Memory mang qua segment")]:
    ax.plot(range(1, N_SEGS + 1), per_seg[k], marker="o", label=lab)
ax.set_xlabel("Segment"); ax.set_ylabel("Loss"); ax.legend(); ax.set_title("Loss theo segment trên tập đánh giá")
plt.tight_layout(); plt.show()

## 6. Lúc test: trọng số thay đổi thế nào khi token đi vào

Mọi tham số đã train được **đóng băng**. Hàm `trace_run` cho văn bản đi qua từng segment và ghi lại:
- khoảng cách của fast-weight so với lúc bắt đầu, `‖M_t − M_0‖ / ‖M_0‖`, và độ lớn mỗi bước cập nhật;
- *surprise* (loss của memory trên từng token) và θ;
- checksum tham số backbone và tham số outer của memory **trước/sau** để chứng minh chỉ fast-weight đổi.

Một Transformer thường có `‖W_t − W_0‖ = 0` trong suốt quá trình (đường phẳng); bộ nhớ của nó là KV cache và tăng tuyến tính theo độ dài.

In [ ]:
def checksum(module):
    return float(sum(p.detach().double().abs().sum().item() for p in module.parameters()))

@torch.no_grad()
def trace_run(model, ids):
    ids = ids.to(device)
    before = (checksum(model.lm), checksum(model.mem) + checksum(model.out))
    model.reset(); model.enabled = True; model.trace = True; model.calls, model.w_ref = [], None
    for s in range(0, len(ids), SEG_LEN):
        seg = ids[s:s + SEG_LEN][None]
        model.lm(input_ids=seg, use_cache=False)
    model.trace = False
    after = (checksum(model.lm), checksum(model.mem) + checksum(model.out))
    cat = lambda key, sl: np.concatenate([c[key][sl] for c in model.calls])
    st = model.state
    fast = sum(v.numel() for k, v in st.weights.items() if k.startswith("model.weights"))
    return dict(dist=cat("dist", slice(1, None)), step=cat("step", slice(None)),
                surprise=cat("surprise", slice(None)), theta=cat("theta", slice(None)),
                before=before, after=after, fast_params=fast, n_tokens=len(ids))

def plot_trace(tr):
    cfg = lm.config
    n = tr["n_tokens"]
    xs_chunk = (np.arange(len(tr["dist"])) + 1) * MEM_CHUNK
    fig, ax = plt.subplots(2, 2, figsize=(11, 7))
    ax[0, 0].plot(xs_chunk, tr["dist"], label="Titans: fast-weight của memory")
    ax[0, 0].plot(xs_chunk, np.zeros_like(xs_chunk, dtype=float), "--", label="Transformer thường: trọng số cố định")
    ax[0, 0].set_title("‖M_t − M_0‖ / ‖M_0‖ theo số token"); ax[0, 0].set_xlabel("token"); ax[0, 0].legend()
    k = max(1, MEM_CHUNK)
    sm = np.convolve(tr["surprise"], np.ones(k) / k, mode="valid")
    ax[0, 1].plot(np.arange(len(sm)) + k, sm, label="surprise (loss memory, làm mượt)")
    ax[0, 1].set_xlabel("token"); ax[0, 1].set_ylabel("surprise")
    a2 = ax[0, 1].twinx(); a2.plot(xs_chunk[:len(tr["step"])], tr["step"], color="tab:red", alpha=.6, label="‖ΔM‖ mỗi chunk")
    ax[0, 1].set_title("Surprise và độ lớn cập nhật"); ax[0, 1].legend(loc="upper left"); a2.legend(loc="upper right")
    ax[1, 0].plot(tr["theta"]); ax[1, 0].set_title("θ_t (learning rate trong inner loop)"); ax[1, 0].set_xlabel("token")
    head_dim = cfg.hidden_size // cfg.num_attention_heads
    toks = np.linspace(0, max(n, 4096), 50)
    kv_mb = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * toks * 2 / 1e6   # fp16
    mem_mb = np.full_like(toks, tr["fast_params"] * 4 / 1e6)
    ax[1, 1].plot(toks, kv_mb, label="Transformer: KV cache (tăng theo độ dài)")
    ax[1, 1].plot(toks, mem_mb, label="Titans: fast-weight memory (cố định)")
    ax[1, 1].set_title("Kích thước bộ nhớ (MB)"); ax[1, 1].set_xlabel("token"); ax[1, 1].legend()
    plt.tight_layout(); plt.show()
    print(f"Checksum backbone  trước: {tr['before'][0]:.6f} | sau: {tr['after'][0]:.6f} | đổi: {tr['after'][0] != tr['before'][0]}")
    print(f"Checksum outer mem trước: {tr['before'][1]:.6f} | sau: {tr['after'][1]:.6f} | đổi: {tr['after'][1] != tr['before'][1]}")
    print(f"Fast-weight đã dịch chuyển tới {tr['dist'][-1]:.4f} lần chuẩn ban đầu (0 = không đổi).")

demo_ids = eval_docs[0][: SEG_LEN * N_SEGS]
tr = trace_run(model, demo_ids)
plot_trace(tr)

## 7. Token nào được hưởng lợi từ memory

Tô màu từng token theo `loss(không có quá khứ) − loss(có memory)`. Xanh = memory giúp dự đoán tốt hơn, đỏ = tệ hơn. Chỉ có ý nghĩa từ segment thứ hai trở đi.

In [ ]:
@torch.no_grad()
def token_losses(model, ids, enabled=True):
    ids = ids.to(device)
    model.reset(); model.enabled = enabled
    losses, tids = [], []
    for s in range(0, len(ids), SEG_LEN):
        seg = ids[s:s + SEG_LEN][None]
        if seg.size(1) < 2:
            break
        logits = model.lm(input_ids=seg, use_cache=False).logits
        l = F.cross_entropy(logits[0, :-1].float(), seg[0, 1:], reduction="none")
        losses.append(l.cpu().numpy()); tids.append(seg[0, 1:].cpu().numpy())
    model.enabled = True
    return np.concatenate(losses), np.concatenate(tids)

def show_gain(model, ids, start_seg=1, max_tokens=400):
    l_mem, tids = token_losses(model, ids, True)
    l_base, _ = token_losses(model, ids, False)
    gain = (l_base - l_mem)
    lo = start_seg * (SEG_LEN - 1)
    gain, tids = gain[lo:lo + max_tokens], tids[lo:lo + max_tokens]
    mx = max(1e-6, float(np.percentile(np.abs(gain), 95)))
    parts = []
    for t, g in zip(tids, gain):
        a = min(1.0, abs(float(g)) / mx)
        col = f"rgba(0,160,0,{a:.2f})" if g > 0 else f"rgba(220,0,0,{a:.2f})"
        parts.append(f'<span style="background:{col}">{_html.escape(decode([int(t)]))}</span>')
    display(HTML("".join(parts)))
    print(f"loss trung bình (đoạn hiển thị): backbone thuần {l_base[lo:lo+max_tokens].mean():.3f} | có memory {l_mem[lo:lo+max_tokens].mean():.3f}")

show_gain(model, demo_ids)

## 8. Kiểm chứng bằng sự kiện tự bịa (đáp án do bạn biết trước)

Tạo một "định luật" chưa tồn tại trong dữ liệu nào, chèn vào đầu, rồi thêm nhiều văn bản nhiễu dài hơn cửa sổ, cuối cùng hỏi lại. So ba cách:
1. **Cửa sổ thường:** backbone chỉ thấy câu hỏi (sự kiện nằm ngoài cửa sổ) — cận dưới.
2. **Titans memory:** sự kiện + nhiễu đi qua memory, rồi mới hỏi.
3. **Toàn bộ ngữ cảnh trong cửa sổ:** Qwen thấy hết (cận trên, chỉ khả thi khi vừa context).

Đáp án chuẩn do bạn gõ nên có thể đối chiếu trực tiếp; kết quả là xác suất mô hình gán cho đáp án đúng. Nếu memory chưa đủ mạnh, kết quả sẽ không tốt hơn cửa sổ thường — hãy báo đúng như vậy.

In [ ]:
FACT   = "Theo ghi chép của phòng thí nghiệm Delta, định luật Zorvan phát biểu rằng hệ số truyền nhiệt của hợp kim Vexa bằng 7,31 đơn vị."
PROMPT = "Câu hỏi: Theo định luật Zorvan, hệ số truyền nhiệt của hợp kim Vexa bằng bao nhiêu? Trả lời: Hệ số bằng"
ANSWER = " 7,31"
FILLER_TOKENS = SEG_LEN * (N_SEGS - 1)

@torch.no_grad()
def answer_score(model, context_ids, prompt_ids, answer_ids, mode):
    ans = torch.tensor(answer_ids, device=device)
    if mode == "titans":
        model.reset(); model.enabled = True
        for s in range(0, len(context_ids), SEG_LEN):
            model.lm(input_ids=context_ids[s:s + SEG_LEN][None].to(device), use_cache=False)
        seq = torch.cat([torch.tensor(prompt_ids, device=device), ans])
    elif mode == "window":
        model.enabled = False; model.reset()
        seq = torch.cat([torch.tensor(prompt_ids, device=device), ans])
    else:  # "full"
        model.enabled = False; model.reset()
        seq = torch.cat([context_ids.to(device), torch.tensor(prompt_ids, device=device), ans])
    logits = model.lm(input_ids=seq[None], use_cache=False).logits[0].float()
    model.enabled = True
    n = len(answer_ids)
    logp = F.log_softmax(logits[-n - 1:-1], dim=-1)
    tok_lp = logp[torch.arange(n), ans]
    top1 = (logp.argmax(-1) == ans).all().item()
    return float(tok_lp.sum()), float(tok_lp.exp().prod()), bool(top1)

fact_ids   = torch.tensor(encode(FACT + "\n"))
context    = torch.cat([fact_ids, eval_docs[0][:FILLER_TOKENS]])
p_ids, a_ids = encode(PROMPT), encode(ANSWER)
print("Độ dài ngữ cảnh (token):", len(context), "| cửa sổ backbone:", SEG_LEN)
rows = []
for mode, name in [("window", "Cửa sổ thường (không thấy sự kiện)"), ("titans", "Titans memory"), ("full", "Toàn bộ ngữ cảnh trong cửa sổ")]:
    lp, pr, ok = answer_score(model, context, p_ids, a_ids, mode)
    rows.append((name, lp, pr, ok))
    print(f"{name:42s} | log P(đáp án) = {lp:8.3f} | P = {pr:.4g} | top-1 đúng: {ok}")

## 9. Hỏi trên tài liệu của bạn (sách / chương vật lý)

Dán văn bản vào `DOC_TEXT`, đặt `PROMPT2`/`ANSWER2` là câu điền khuyết lấy từ chính tài liệu (đáp án có sẵn trong sách để đối chiếu). Nhớ chọn chi tiết mà Qwen không thể biết trước (số liệu riêng, quy ước riêng), nếu không tín hiệu của memory sẽ lẫn với kiến thức có sẵn của Qwen.

In [ ]:
DOC_TEXT = ""      # dán một chương sách (khoảng 50-100KB) vào đây
PROMPT2  = ""      # ví dụ: "Theo giáo trình, đơn vị của đại lượng X là"
ANSWER2  = ""      # ví dụ: " kilogam"

if DOC_TEXT and PROMPT2 and ANSWER2:
    ctx = torch.tensor(encode(DOC_TEXT))
    p2, a2 = encode(PROMPT2), encode(ANSWER2)
    print("Độ dài tài liệu (token):", len(ctx))
    tr2 = trace_run(model, ctx)
    plot_trace(tr2)
    for mode, name in [("window", "Cửa sổ thường"), ("titans", "Titans memory"), ("full", "Toàn bộ ngữ cảnh")]:
        try:
            lp, pr, ok = answer_score(model, ctx, p2, a2, mode)
            print(f"{name:20s} | log P(đáp án) = {lp:8.3f} | P = {pr:.4g} | top-1 đúng: {ok}")
        except torch.cuda.OutOfMemoryError:
            print(f"{name:20s} | hết bộ nhớ GPU (tài liệu quá dài cho chế độ này)")
else:
    print("Chưa nhập DOC_TEXT/PROMPT2/ANSWER2, bỏ qua mục này.")

## 10. Xuất file cho webapp

File `titans_vi_memory.pt` (không kèm optimizer, vài chục MB) chứa trọng số `mem`, `out`, `gate` và cấu hình. Tải file này từ tab *Output* của Kaggle để dùng cho webapp ở bước tiếp theo.

In [ ]:
final = os.path.join(CKPT_DIR, "titans_vi_memory.pt")
print(final, "| kích thước MB:", round(os.path.getsize(final) / 1e6, 1))